In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Combine after-review result tables into one reviewer-oriented table.

Strict-patristic version
------------------------
This version reads THREE files for EACH method:

1. Matrix-completion summary:
   *_summary_formatted_by_missingness.csv

2. Tree-topology summary:
   tree_topology_comparison/tree_topology_summary_meanstd.csv

3. ML-topology branch-length summary:
   ml_branch_length_comparison/ml_branch_length_summary_meanstd.csv

Important methodological choice
-------------------------------
The main-table patristic columns are NOT mixed anymore.

Main columns:
    pat. RMSE
    pat. Spear.

use only the NJ-tree-vs-reference-ML-tree patristic comparison:
    nj_vs_ML_pat_RMSE
    nj_vs_ML_pat_Spearman

from:
    tree_topology_comparison/tree_topology_summary_meanstd.csv

The fixed-ML-topology fitted-patristic quantities:
    fit_pat_RMSE
    fit_pat_Spearman

from:
    ml_branch_length_comparison/ml_branch_length_summary_meanstd.csv

are kept separately as diagnostics in the numeric CSV and in the reviewer-audit
LaTeX table. They are not used as fallback values for the main patristic columns.

Warm-start comparison
---------------------
The folder Hyb-Adam-UM-warmstart is included as a separate method row
(Hyb-Adam-UM+WS), so it can be compared directly against the original
Hyb-Adam-UM and all baseline methods.

Outputs
-------
combined_reviewer_tables/
    combined_all_metrics_numeric.csv
    combined_file_inventory.csv
    combined_main_results_table.tex
    combined_reviewer_audit_table.tex
"""

from __future__ import annotations

import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

OUT_DIR = Path("combined_reviewer_tables")
OUT_DIR.mkdir(exist_ok=True)

MISSING_LEVELS = [30, 50, 65, 85]

# Main experiment is 15 taxa, so overline{Delta} must be Delta_total / C(15,3).
# This is used only as a fallback when a source table has Delta_total but lacks
# an explicit Delta_per_triangle column.
N_TAXA_FOR_DELTA = 15
N_TRIPLETS_FOR_DELTA = math.comb(N_TAXA_FOR_DELTA, 3)

# The reviewer protocol uses 30 masks/runs per missingness level.
# Success is displayed as x/30, where x is computed from n_success when available,
# or inferred from n_failed / available metrics when possible.
DEFAULT_TOTAL_RUNS = 30


@dataclass(frozen=True)
class MethodSpec:
    key: str
    latex: str
    aliases: Sequence[str]

    matrix_files: Sequence[Path]
    topology_files: Sequence[Path]
    branch_files: Sequence[Path]


METHODS: List[MethodSpec] = [


    MethodSpec(
        key="Hyb-Adam-UM-improved",
        latex=r"Hyb-Adam-UM",
        aliases=(
            "Hyb-Adam-UM-warmstart", "Hyb Adam UM warmstart",
            "Hyb-Adam-UM-WS", "HybAdamUMWarmstart", "HybAdamUMWS",
            "Hyb-Adam-UM+WS", "Hyb Adam UM WS",
            # Many warm-start runs still write the base method name inside the CSV.
            # This is safe because this MethodSpec reads only the warm-start folder.
            "Hyb-Adam-UM", "HybAdamUM", "Hyb Adam UM",
        ),
        matrix_files=(
            # Most likely: same internal output structure, only the parent folder changed.
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_outputs/tables/hyb_adam_um_summary_formatted_by_missingness.csv"),
            # Alternative names in case the improved code writes warm-start-specific folders/files.
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_warmstart_outputs/tables/hyb_adam_um_warmstart_summary_formatted_by_missingness.csv"),
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_warmstart_outputs/tables/hyb_adam_um_summary_formatted_by_missingness.csv"),
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_ws_outputs/tables/hyb_adam_um_ws_summary_formatted_by_missingness.csv"),
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_ws_outputs/tables/hyb_adam_um_summary_formatted_by_missingness.csv"),
        ),
        topology_files=(
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_outputs/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_warmstart_outputs/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_ws_outputs/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
        ),
        branch_files=(
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_outputs/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_warmstart_outputs/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
            Path("Hyb-Adam-UM-lr002-5ksteps-1start/hyb_adam_um_ws_outputs/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
        ),
    ),
    
    MethodSpec(
        key="MW-proj",
        latex=r"MW$^\star$-proj",
        aliases=("MW-proj", "MWProj", "MW proj", "MW*-proj", "MWstar", "MW-Proj"),
        matrix_files=(
            Path("MW-proj/mw_proj_outputs/tables/mw_proj_summary_formatted_by_missingness.csv"),
        ),
        topology_files=(
            Path("MW-proj/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
        ),
        branch_files=(
            Path("MW-proj/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
        ),
    ),





    MethodSpec(
        key="NJstar-STRICT",
        latex=r"NJ$^\star$-proj",
        aliases=(
            "NJstar-STRICT", "NJstar_STRICT", "NJStarStrict",
            "NJ-proj", "NJstar", "NJ-star", "NJ*", "NJ⋆",
            "NJstar-strict", "NJ-star-strict",
        ),
        matrix_files=(
            Path("NJ-proj/njstar_strict_outputs/tables/njstar_strict_summary_formatted_by_missingness.csv"),
            Path("NJ-proj-afterreview/njstar_strict_outputs/tables/njstar_strict_summary_formatted_by_missingness.csv"),
        ),
        topology_files=(
            Path("NJ-proj/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
            Path("NJ-proj-afterreview/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
        ),
        branch_files=(
            Path("NJ-proj/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
            Path("NJ-proj-afterreview/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
        ),
    ),

    MethodSpec(
        key="LRMC-SoftImpute",
        latex="LRMC",
        aliases=("LRMC", "LRMC-SoftImpute", "LRMC SoftImpute", "SoftImpute", "Soft-Impute"),
        matrix_files=(
            Path("LRMC/lrmc_softimpute_outputs/tables/lrmc_softimpute_summary_formatted_by_missingness.csv"),
        ),
        topology_files=(
            Path("LRMC/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
        ),
        branch_files=(
            Path("LRMC/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
        ),
    ),

    MethodSpec(
        key="KNN-Impute",
        latex="KNN-impute",
        aliases=("KNN-Impute", "KNN-impute", "KNN", "KNNImpute", "KNNimpute", "KNN-Imputer"),
        matrix_files=(
            Path("KNN-impute/knn_impute_outputs/tables/knn_impute_summary_formatted_by_missingness.csv"),
        ),
        topology_files=(
            Path("KNN-impute/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
        ),
        branch_files=(
            Path("KNN-impute/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
        ),
    ),

    MethodSpec(
        key="MDS-SMACOF",
        latex="MDS-SMACOF",
        aliases=(
            "MDS-SMACOF",
            "MDS SMACOF",
            "MDSSMACOF",
            "MDS-Smacof",
            "mds-smacof",
            "MDS",
            "SMACOF",
        ),
        matrix_files=(
            Path("MDS-SMACOF/mds_smacof_outputs/tables/mds_smacof_summary_formatted_by_missingness.csv"),
            Path("./MDS-SMACOF/mds_smacof_outputs/tables/mds_smacof_summary_formatted_by_missingness.csv"),
            Path("/home/user/bioinformatics/!after review -ready/MDS-SMACOF/mds_smacof_outputs/tables/mds_smacof_summary_formatted_by_missingness.csv"),
        ),
        topology_files=(
            Path("MDS-SMACOF/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
            Path("./MDS-SMACOF/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
            Path("/home/user/bioinformatics/!after review -ready/MDS-SMACOF/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
        ),
        branch_files=(
            Path("MDS-SMACOF/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
            Path("./MDS-SMACOF/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
            Path("/home/user/bioinformatics/!after review -ready/MDS-SMACOF/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
        ),
    ),
    # MethodSpec(
    #     key="Ultrametric-fill",
    #     latex="Ultrametric-fill",
    #     aliases=(
    #         "Ultrametric-fill",
    #         "UltrametricFill",
    #         "Ultrametric fill",
    #         "Ultrametric-Fill",
    #         "Ultrametric_Fill",
    #         "ultrametric-fill",
    #         "ultrametric_fill",
    #     ),
    #     matrix_files=(
    #         Path("Ultrametric-fill/ultrametric_fill_outputs/tables/ultrametric_fill_summary_formatted_by_missingness.csv"),
    #         Path("./Ultrametric-fill/ultrametric_fill_outputs/tables/ultrametric_fill_summary_formatted_by_missingness.csv"),
    #         Path("/Ultrametric-fill/ultrametric_fill_outputs/tables/ultrametric_fill_summary_formatted_by_missingness.csv"),
    #         Path("/home/user/bioinformatics/!after review/Ultrametric-fill/ultrametric_fill_outputs/tables/ultrametric_fill_summary_formatted_by_missingness.csv"),

    #         # fallback if you run from inside the Ultrametric-fill folder
    #         Path("ultrametric_fill_outputs/tables/ultrametric_fill_summary_formatted_by_missingness.csv"),
    #         Path("./ultrametric_fill_outputs/tables/ultrametric_fill_summary_formatted_by_missingness.csv"),
    #         Path("/mnt/data/ultrametric_fill_outputs/tables/ultrametric_fill_summary_formatted_by_missingness.csv"),
    #     ),
    #     topology_files=(
    #         Path("Ultrametric-fill/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
    #         Path("Ultrametric-fill/ultrametric_fill_outputs/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
    #         Path("/Ultrametric-fill/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
    #         Path("/Ultrametric-fill/ultrametric_fill_outputs/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
    #         Path("/home/user/bioinformatics/!after review/Ultrametric-fill/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
    #         Path("/home/user/bioinformatics/!after review/Ultrametric-fill/ultrametric_fill_outputs/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv"),
    #     ),
    #     branch_files=(
    #         Path("Ultrametric-fill/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
    #         Path("Ultrametric-fill/ultrametric_fill_outputs/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
    #         Path("/Ultrametric-fill/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
    #         Path("/Ultrametric-fill/ultrametric_fill_outputs/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
    #         Path("/home/user/bioinformatics/!after review/Ultrametric-fill/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
    #         Path("/home/user/bioinformatics/!after review/Ultrametric-fill/ultrametric_fill_outputs/ml_branch_length_comparison_heterogeneous_primate/ml_branch_length_summary_meanstd.csv"),
    #     ),
    # ),

]


# ============================================================
# Parsing helpers
# ============================================================

PM_REPLACEMENTS = {
    r"\pm": "±",
    "+/-": "±",
    "+-": "±",
    "−": "-",
    "–": "-",
    "—": "-",
}


def norm_text(s: str) -> str:
    """Normalize text for robust column/method matching."""
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())


def latex_escape_text(s: str) -> str:
    return (
        str(s)
        .replace("_", r"\_")
        .replace("%", r"\%")
        .replace("&", r"\&")
    )


def choose_existing_path(paths: Sequence[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None


def clean_cell(x) -> str:
    """Clean a CSV cell that may contain LaTeX formatting or mean ± std text."""
    if pd.isna(x):
        return ""

    s = str(x).strip()
    for a, b in PM_REPLACEMENTS.items():
        s = s.replace(a, b)

    s = s.replace(r"\best", "")
    s = s.replace(r"\textbf", "")
    s = s.replace("$", "")
    s = s.replace("{", "")
    s = s.replace("}", "")
    s = s.replace(",", "")
    s = s.strip()
    return s


def first_float(s: str) -> float:
    """Extract the first floating-point number from a string."""
    s = clean_cell(s)
    if not s:
        return np.nan

    if "N/A" in s.upper() or s.lower() in {"nan", "none", "-"}:
        return np.nan

    # Handle simple 10^k forms.
    m_pow = re.search(r"10\s*\^\s*([+-]?\d+)", s)
    if m_pow:
        return float(10.0 ** int(m_pow.group(1)))

    m = re.search(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?", s)
    if not m:
        return np.nan
    return float(m.group(0))


def parse_mean_std(x) -> Tuple[float, float, str]:
    """
    Parse either:
        mean
    or:
        mean ± std

    Returns:
        mean, std, raw_original_string
    """
    raw = "" if pd.isna(x) else str(x).strip()
    s = clean_cell(x)

    if not s or "N/A" in s.upper() or s.lower() in {"nan", "none", "-"}:
        return np.nan, np.nan, raw

    if "±" in s:
        left, right = s.split("±", 1)
        return first_float(left), first_float(right), raw

    return first_float(s), np.nan, raw


def detect_pct_column(df: pd.DataFrame) -> str:
    candidates = ["pct_missing", "% Missing", "Missing", "p", "missingness", "missingness_level"]
    for c in candidates:
        if c in df.columns:
            return c

    for c in df.columns:
        if "missing" in norm_text(c):
            return c

    raise ValueError(f"Cannot find missingness column. Columns: {list(df.columns)}")


def normalize_pct_value(x) -> int:
    if pd.isna(x):
        return -1

    s = str(x).strip().replace("%", "")
    v = first_float(s)
    if not np.isfinite(v):
        return -1

    if 0 < v < 1:
        v *= 100
    return int(round(v))


def filter_method_rows(df: pd.DataFrame, aliases: Sequence[str]) -> pd.DataFrame:
    """
    Keep rows matching the method aliases.

    If there is no method column, assume the file is already method-specific.
    If there is exactly one method in the file, accept it even if the spelling differs.
    """
    if "method" not in df.columns:
        return df.copy()

    aliases_norm = {norm_text(a) for a in aliases}
    method_norm = df["method"].map(norm_text)
    mask = method_norm.isin(aliases_norm)

    if mask.any():
        return df[mask].copy()

    if df["method"].nunique(dropna=True) == 1:
        return df.copy()

    return df.iloc[0:0].copy()


def load_summary_by_pct(path: Optional[Path], aliases: Sequence[str]) -> Dict[int, pd.Series]:
    """Load one summary CSV and return rows indexed by missingness percentage."""
    if path is None or not path.exists():
        return {}

    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]

    df = filter_method_rows(df, aliases)
    if df.empty:
        return {}

    pct_col = detect_pct_column(df)
    df["_pct"] = df[pct_col].map(normalize_pct_value)
    df = df[df["_pct"].isin(MISSING_LEVELS)].copy()

    out: Dict[int, pd.Series] = {}
    for pct, sub in df.groupby("_pct", sort=True):
        out[int(pct)] = sub.iloc[0]
    return out


def extract_metric(row: Optional[pd.Series], candidates: Sequence[str]) -> Tuple[float, float, str, str]:
    """
    Extract a metric from a row using a list of possible column names.

    Returns:
        mean, std, raw_string, source_column
    """
    if row is None:
        return np.nan, np.nan, "", ""

    col_map = {norm_text(c): c for c in row.index}

    for cand in candidates:
        c = col_map.get(norm_text(cand))
        if c is None:
            continue

        mean, std, raw = parse_mean_std(row[c])
        if np.isfinite(mean) or raw not in {"", "nan", "N/A"}:
            return mean, std, raw, c

    return np.nan, np.nan, "", ""


def set_metric(
    rec: Dict[str, object],
    prefix: str,
    row: Optional[pd.Series],
    candidates: Sequence[str],
) -> None:
    mean, std, raw, col = extract_metric(row, candidates)
    rec[f"{prefix}_mean"] = mean
    rec[f"{prefix}_std"] = std
    rec[f"{prefix}_raw"] = raw
    rec[f"{prefix}_source_col"] = col


def set_metric_from_sources(
    rec: Dict[str, object],
    prefix: str,
    sources: Sequence[Tuple[str, Optional[pd.Series], Sequence[str]]],
) -> None:
    """
    Extract a metric from ordered sources.

    In this strict-patristic script, the main patristic metrics are passed with
    only one source: topology_nj_vs_ml. Therefore no fixed-topology fallback is
    used for the main patristic columns.
    """
    for source_name, row, candidates in sources:
        mean, std, raw, col = extract_metric(row, candidates)
        if np.isfinite(mean):
            rec[f"{prefix}_mean"] = mean
            rec[f"{prefix}_std"] = std
            rec[f"{prefix}_raw"] = raw
            rec[f"{prefix}_source"] = source_name
            rec[f"{prefix}_source_col"] = col
            return

    rec[f"{prefix}_mean"] = np.nan
    rec[f"{prefix}_std"] = np.nan
    rec[f"{prefix}_raw"] = ""
    rec[f"{prefix}_source"] = ""
    rec[f"{prefix}_source_col"] = ""


def set_delta_per_triangle_metric(
    rec: Dict[str, object],
    prefix: str,
    row: Optional[pd.Series],
    *,
    n_triplets: int = N_TRIPLETS_FOR_DELTA,
) -> None:
    """
    Store the table's overline{Delta} value as the true triplet-averaged delta.

    Priority:
    1. Use explicit Delta_per_triangle/Delta_per_triplet columns.
    2. If only a total Delta column is available, divide mean and std by C(15,3).
    3. Do NOT use Delta_normalized_completed here, because in the Hyb-Adam code
       that column is the log-compressed normalized delta, not Delta/C(n,3).
    """
    per_triangle_candidates = [
        "Delta_per_triangle_completed",
        "Delta_per_triplet_completed",
        "Delta_per_triangle",
        "Delta_per_triplet",
        "Delta_mean_per_triangle",
        "Delta_mean_per_triplet",
        "Delta_avg_triangle",
        "Delta_avg_triplet",
    ]

    mean, std, raw, col = extract_metric(row, per_triangle_candidates)
    if np.isfinite(mean):
        rec[f"{prefix}_mean"] = mean
        rec[f"{prefix}_std"] = std
        rec[f"{prefix}_raw"] = raw
        rec[f"{prefix}_source_col"] = col
        rec[f"{prefix}_source"] = "per_triangle"
        return

    total_candidates = [
        "Delta_total_completed",
        "Delta_total",
        "Delta_sum_completed",
        "Delta_sum",
    ]

    mean, std, raw, col = extract_metric(row, total_candidates)
    if np.isfinite(mean) and n_triplets > 0:
        rec[f"{prefix}_mean"] = mean / n_triplets
        rec[f"{prefix}_std"] = std / n_triplets if np.isfinite(std) else np.nan
        rec[f"{prefix}_raw"] = raw
        rec[f"{prefix}_source_col"] = f"{col}/C({N_TAXA_FOR_DELTA},3)"
        rec[f"{prefix}_source"] = "total_divided_by_triplets"
        return

    rec[f"{prefix}_mean"] = np.nan
    rec[f"{prefix}_std"] = np.nan
    rec[f"{prefix}_raw"] = ""
    rec[f"{prefix}_source_col"] = ""
    rec[f"{prefix}_source"] = ""


# ============================================================
# Build combined numeric table
# ============================================================

def build_combined_table() -> Tuple[pd.DataFrame, pd.DataFrame]:
    records: List[Dict[str, object]] = []
    inventory_rows: List[Dict[str, object]] = []

    for spec in METHODS:
        matrix_path = choose_existing_path(spec.matrix_files)
        topology_path = choose_existing_path(spec.topology_files)
        branch_path = choose_existing_path(spec.branch_files)

        inventory_rows.extend([
            {
                "method": spec.key,
                "file_type": "matrix",
                "found": matrix_path is not None,
                "used_path": str(matrix_path) if matrix_path else "",
                "candidate_paths": " | ".join(str(p) for p in spec.matrix_files),
            },
            {
                "method": spec.key,
                "file_type": "topology",
                "found": topology_path is not None,
                "used_path": str(topology_path) if topology_path else "",
                "candidate_paths": " | ".join(str(p) for p in spec.topology_files),
            },
            {
                "method": spec.key,
                "file_type": "branch",
                "found": branch_path is not None,
                "used_path": str(branch_path) if branch_path else "",
                "candidate_paths": " | ".join(str(p) for p in spec.branch_files),
            },
        ])

        matrix_by_pct = load_summary_by_pct(matrix_path, spec.aliases)
        topo_by_pct = load_summary_by_pct(topology_path, spec.aliases)
        branch_by_pct = load_summary_by_pct(branch_path, spec.aliases)

        for pct in MISSING_LEVELS:
            mrow = matrix_by_pct.get(pct)
            trow = topo_by_pct.get(pct)
            brow = branch_by_pct.get(pct)

            rec: Dict[str, object] = {
                "pct_missing": pct,
                "method_key": spec.key,
                "method_latex": spec.latex,
                "matrix_summary_path": str(matrix_path) if matrix_path else "",
                "topology_summary_path": str(topology_path) if topology_path else "",
                "branch_summary_path": str(branch_path) if branch_path else "",
                "matrix_row_available": mrow is not None,
                "topology_row_available": trow is not None,
                "branch_row_available": brow is not None,
            }

            # -------------------------
            # Matrix-level metrics
            # -------------------------
            set_metric(rec, "rmse_miss", mrow, ["RMSE_miss", "RMSE"])
            set_metric(rec, "mae_miss", mrow, ["MAE_miss", "MAE"])
            set_metric(rec, "pearson_miss", mrow, ["Pearson_miss", "Pearson"])
            set_metric(rec, "spearman_miss", mrow, ["Spearman_miss", "Spearman"])
            set_metric(rec, "runtime_seconds", mrow, ["runtime_seconds", "runtime", "time_seconds", "Time(s)", "Time"])
            # Table column overline{Delta}(Dhat): true delta per triangle.
            # Do not use Delta_normalized_completed here: in the Hyb-Adam code it is
            # log-compressed and is not Delta_total / number_of_triplets.
            set_delta_per_triangle_metric(rec, "delta_norm", mrow)

            # -------------------------
            # Reviewer diagnostics
            # -------------------------
            set_metric(rec, "n_total", mrow, ["n_total", "n_replicates", "n_files"])
            set_metric(rec, "n_success", mrow, ["n_success"])
            set_metric(rec, "n_failed", mrow, ["n_failed"])
            set_metric(rec, "n_missing", mrow, ["n_missing"])
            set_metric(rec, "n_observed", mrow, ["n_observed"])
            set_metric(rec, "optimized_variables", mrow, ["optimized_variables"])
            set_metric(rec, "imputed_variables", mrow, ["imputed_variables"])
            set_metric(rec, "epochs_used", mrow, ["epochs_used"])
            set_metric(rec, "convergence_epoch", mrow, ["convergence_epoch"])
            set_metric(rec, "iters_used", mrow, ["iters_used", "iterations", "n_iter", "n_iterations"])
            set_metric(rec, "final_rank", mrow, ["final_rank"])
            set_metric(rec, "tree_edges_final", mrow, ["tree_edges_final"])
            set_metric(rec, "joins_performed", mrow, ["joins_performed"])

            # -------------------------
            # Tree-topology metrics
            # -------------------------
            set_metric(rec, "nrf", trow, ["nRF", "RF_norm", "RF_normalized"])
            set_metric(rec, "rf", trow, ["RF"])
            set_metric(rec, "split_f1", trow, ["split_F1"])
            set_metric(rec, "split_precision", trow, ["split_precision"])
            set_metric(rec, "split_recall", trow, ["split_recall"])

            # Explicit NJ-tree-vs-ML-tree patristic metrics.
            set_metric(rec, "nj_pat_rmse", trow, ["nj_vs_ML_pat_RMSE"])
            set_metric(rec, "nj_pat_mae", trow, ["nj_vs_ML_pat_MAE"])
            set_metric(rec, "nj_pat_spearman", trow, ["nj_vs_ML_pat_Spearman"])
            set_metric(rec, "nj_pat_pearson", trow, ["nj_vs_ML_pat_Pearson"])

            # -------------------------
            # ML fixed-topology branch-length metrics
            # -------------------------
            set_metric(rec, "edge_bl_rmse", brow, ["edge_BL_RMSE"])
            set_metric(rec, "edge_bl_mae", brow, ["edge_BL_MAE"])
            set_metric(rec, "edge_bl_rel_rmse", brow, ["edge_BL_rel_RMSE"])
            set_metric(rec, "edge_bl_spearman", brow, ["edge_BL_Spearman"])
            set_metric(rec, "edge_bl_pearson", brow, ["edge_BL_Pearson"])
            set_metric(rec, "terminal_bl_rmse", brow, ["terminal_BL_RMSE"])
            set_metric(rec, "internal_bl_rmse", brow, ["internal_BL_RMSE"])

            # Fixed-ML-topology fitted-patristic diagnostics.
            # These are NOT used as fallback for the main patristic columns.
            set_metric(rec, "fit_pat_rmse", brow, ["fit_pat_RMSE"])
            set_metric(rec, "fit_pat_mae", brow, ["fit_pat_MAE"])
            set_metric(rec, "fit_pat_spearman", brow, ["fit_pat_Spearman"])
            set_metric(rec, "fit_pat_pearson", brow, ["fit_pat_Pearson"])
            set_metric(rec, "tree_length_rel_error", brow, ["tree_length_rel_error"])

            # -------------------------
            # Main-table patristic metrics: STRICTLY topology NJ-vs-ML only.
            # -------------------------
            set_metric_from_sources(
                rec,
                "pat_rmse",
                [
                    ("topology_nj_vs_ml", trow, ["nj_vs_ML_pat_RMSE"]),
                ],
            )
            set_metric_from_sources(
                rec,
                "pat_spearman",
                [
                    ("topology_nj_vs_ml", trow, ["nj_vs_ML_pat_Spearman"]),
                ],
            )

            records.append(rec)

    return pd.DataFrame(records), pd.DataFrame(inventory_rows)


# ============================================================
# LaTeX formatting
# ============================================================

LOWER_IS_BETTER = {
    "rmse_miss",
    "mae_miss",
    "delta_norm",
    "runtime_seconds",
    "nrf",
    "pat_rmse",
    "edge_bl_rmse",
    "fit_pat_rmse",
    "nj_pat_rmse",
}

HIGHER_IS_BETTER = {
    "pearson_miss",
    "spearman_miss",
    "pat_spearman",
    "edge_bl_spearman",
    "fit_pat_spearman",
    "nj_pat_spearman",
    "split_f1",
}


def best_mask(df: pd.DataFrame, metric: str) -> Dict[Tuple[int, str], bool]:
    out: Dict[Tuple[int, str], bool] = {}
    mean_col = f"{metric}_mean"

    if mean_col not in df.columns:
        return out

    for pct, sub in df.groupby("pct_missing"):
        vals = sub[["method_key", mean_col]].copy()
        vals = vals[np.isfinite(vals[mean_col].to_numpy(dtype=float))]
        if vals.empty:
            continue

        arr = vals[mean_col].to_numpy(dtype=float)

        if metric in LOWER_IS_BETTER:
            target = np.nanmin(arr)
        elif metric in HIGHER_IS_BETTER:
            target = np.nanmax(arr)
        else:
            continue

        tol = max(1e-12, abs(target) * 1e-10)
        for _, r in vals.iterrows():
            out[(int(pct), str(r["method_key"]))] = abs(float(r[mean_col]) - target) <= tol

    return out


def fmt_number(x: float, decimals: int = 2) -> str:
    if not np.isfinite(x):
        return "N/A"

    if abs(x) >= 1e6:
        exp = int(math.floor(math.log10(abs(x))))
        coeff = x / (10 ** exp)
        if abs(coeff - 1.0) < 0.05:
            return rf"$\approx 10^{{{exp}}}$"
        return rf"${coeff:.1f}\times 10^{{{exp}}}$"

    if 0 < abs(x) < 0.01 and decimals <= 2:
        return r"$<0.01$"

    return f"{x:.{decimals}f}"


def fmt_pm_latex(
    mean: float,
    std: float,
    *,
    decimals: int = 2,
    scale: float = 1.0,
    best: bool = False,
) -> str:
    if not np.isfinite(mean):
        return "N/A"

    m = mean * scale
    s = std * scale if np.isfinite(std) else np.nan

    if np.isfinite(s):
        text = f"{fmt_number(m, decimals)} $\\pm$ {fmt_number(s, decimals)}"
    else:
        text = fmt_number(m, decimals)

    if best:
        return rf"\best{{{text}}}"
    return text


def metric_cell(
    row: pd.Series,
    metric: str,
    best_lookup: Dict[str, Dict[Tuple[int, str], bool]],
    *,
    decimals: int = 2,
    scale: float = 1.0,
) -> str:
    mean = row.get(f"{metric}_mean", np.nan)
    std = row.get(f"{metric}_std", np.nan)
    is_best = best_lookup.get(metric, {}).get((int(row["pct_missing"]), str(row["method_key"])), False)
    return fmt_pm_latex(mean, std, decimals=decimals, scale=scale, best=is_best)


def diag_pair_cell(row: pd.Series, a: str, b: str, decimals: int = 0) -> str:
    av = row.get(f"{a}_mean", np.nan)
    bv = row.get(f"{b}_mean", np.nan)

    if not np.isfinite(av) and not np.isfinite(bv):
        return "N/A"
    if not np.isfinite(av):
        return f"N/A/{fmt_number(bv, decimals)}"
    if not np.isfinite(bv):
        return f"{fmt_number(av, decimals)}/N/A"
    return f"{fmt_number(av, decimals)}/{fmt_number(bv, decimals)}"


def _int_metric_value(row: pd.Series, metric_prefix: str) -> Optional[int]:
    """Return a rounded integer metric value, or None if unavailable."""
    val = row.get(f"{metric_prefix}_mean", np.nan)
    if np.isfinite(val):
        return int(round(float(val)))
    return None


def _row_has_any_finite_metric(row: pd.Series, metric_prefixes: Sequence[str]) -> bool:
    for metric in metric_prefixes:
        val = row.get(f"{metric}_mean", np.nan)
        if np.isfinite(val):
            return True
    return False


def success_cell(row: pd.Series, default_total_runs: int = DEFAULT_TOTAL_RUNS) -> str:
    """
    Display success as x/30, not as a forced constant.

    Rules:
    1. Denominator: use n_total/n_replicates/n_files if present; otherwise use 30.
       This is the only place where the missing denominator is replaced by 30.
    2. Numerator: use n_success if present.
    3. If n_success is absent but n_failed is present, compute n_total - n_failed.
    4. If success metadata is absent but the row has finite matrix metrics, infer that
       all denominator runs contributed to the summary.
    5. If a matrix row exists but all matrix metrics are unavailable, report 0/denom.
       This avoids the wrong NJ-at-85% situation where all metrics are N/A but Succ.
       was incorrectly shown as 30.
    6. If there is no matrix row and no success metadata, report N/A/denom.
    """
    total = _int_metric_value(row, "n_total")
    if total is None or total <= 0:
        total = int(default_total_runs)

    success = _int_metric_value(row, "n_success")
    failed = _int_metric_value(row, "n_failed")

    if success is None and failed is not None:
        success = max(0, total - failed)

    if success is None:
        finite_matrix_metrics = _row_has_any_finite_metric(
            row,
            ["rmse_miss", "mae_miss", "pearson_miss", "spearman_miss"],
        )
        matrix_row_available = bool(row.get("matrix_row_available", False))
        if finite_matrix_metrics:
            success = total
        elif matrix_row_available:
            success = 0

    if success is None:
        return f"N/A/{total}"

    success = max(0, min(int(success), int(total)))
    return f"{success}/{total}"


def variables_cell(row: pd.Series) -> str:
    opt = row.get("optimized_variables_mean", np.nan)
    imp = row.get("imputed_variables_mean", np.nan)

    if np.isfinite(opt) and opt > 0:
        return fmt_number(opt, 0)
    if np.isfinite(imp) and imp > 0:
        return fmt_number(imp, 0)
    if np.isfinite(opt):
        return fmt_number(opt, 0)
    return "N/A"


def epoch_or_iter_cell(row: pd.Series) -> str:
    conv = row.get("convergence_epoch_mean", np.nan)
    epochs = row.get("epochs_used_mean", np.nan)
    iters = row.get("iters_used_mean", np.nan)
    joins = row.get("joins_performed_mean", np.nan)

    if np.isfinite(conv) and conv > 0:
        return fmt_number(conv, 0)
    if np.isfinite(epochs) and epochs > 0:
        return fmt_number(epochs, 0)
    if np.isfinite(iters) and iters > 0:
        return fmt_number(iters, 0)
    if np.isfinite(joins) and joins > 0:
        return fmt_number(joins, 0)
    return "N/A"


def ordered_subtable(df: pd.DataFrame, pct: int) -> pd.DataFrame:
    order = [m.key for m in METHODS]
    sub = df[df["pct_missing"] == pct].copy()
    sub["_order"] = sub["method_key"].map({k: i for i, k in enumerate(order)})
    sub = sub.sort_values("_order").drop(columns=["_order"])
    return sub


def make_latex_main(df: pd.DataFrame) -> str:
    metrics_for_best = [
        "rmse_miss",
        "mae_miss",
        "pearson_miss",
        "spearman_miss",
        "runtime_seconds",
        "nrf",
        "pat_rmse",
        "pat_spearman",
        "edge_bl_rmse",
    ]
    best_lookup = {m: best_mask(df, m) for m in metrics_for_best}

    lines: List[str] = []
    lines.append(r"% Requires: \usepackage{booktabs,multirow,rotating}")
    lines.append(r"% Recommended macro: \newcommand{\best}[1]{\textbf{#1}}")
    lines.append(r"\begin{sidewaystable}[t]")
    lines.append(
        r"\caption{Combined matrix-, topology-, and ML-branch-length performance on "
        r"$15\times 15$ mtDNA distance matrices. Matrix errors are computed only on "
        r"artificially hidden entries. RMSE/MAE, NJ-vs-ML patristic RMSE, and "
        r"branch-length RMSE are reported as $\times 10^{-2}$. Lower is better for "
        r"RMSE/MAE, runtime, RF$_{\mathrm{norm}}$, NJ-vs-ML patristic RMSE, and "
        r"branch-length RMSE; higher is better for correlations. Best values within "
        r"each missingness level are in bold.}"
    )
    lines.append(r"\label{tab:combined_main_results}")
    lines.append(r"\centering")
    lines.append(r"\tiny")
    lines.append(r"\setlength{\tabcolsep}{2pt}")
    lines.append(r"\begin{tabular}{llcccccccccc}")
    lines.append(r"\toprule")
    lines.append(
        r"& & \multicolumn{5}{c}{Matrix metrics} & "
        r"\multicolumn{5}{c}{Tree / branch-length metrics} \\" 
    )
    lines.append(r"\cmidrule(lr){3-7}\cmidrule(lr){8-12}")
    lines.append(
        r"$p$ & Method & RMSE & MAE & Pear. & Spear. & Time(s) & "
        r"RF$_{\mathrm{norm}}$ & NJ pat.\ RMSE & NJ pat.\ Spear. & edge BL RMSE & Succ. \\"
    )
    lines.append(r"\midrule")

    for pct in MISSING_LEVELS:
        sub = ordered_subtable(df, pct)
        nrows = len(sub)

        for i, (_, row) in enumerate(sub.iterrows()):
            pcell = rf"\multirow{{{nrows}}}{{*}}{{{pct}\%}}" if i == 0 else ""
            succ = success_cell(row)

            line = " & ".join([
                pcell,
                row["method_latex"],
                metric_cell(row, "rmse_miss", best_lookup, decimals=2, scale=100.0),
                metric_cell(row, "mae_miss", best_lookup, decimals=2, scale=100.0),
                metric_cell(row, "pearson_miss", best_lookup, decimals=2),
                metric_cell(row, "spearman_miss", best_lookup, decimals=2),
                metric_cell(row, "runtime_seconds", best_lookup, decimals=2),
                metric_cell(row, "nrf", best_lookup, decimals=2),
                metric_cell(row, "pat_rmse", best_lookup, decimals=2, scale=100.0),
                metric_cell(row, "pat_spearman", best_lookup, decimals=2),
                metric_cell(row, "edge_bl_rmse", best_lookup, decimals=2, scale=100.0),
                succ,
            ]) + r" \\"
            lines.append(line)

        if pct != MISSING_LEVELS[-1]:
            lines.append(r"\midrule")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\vspace{2pt}")
    lines.append(
        r"\parbox{\textwidth}{\scriptsize "
        r"Succ. is displayed as successful/evaluated runs in the 30-run protocol; "
        r"when the denominator is absent from a source file, it is set to 30. "
        r"The numerator is computed from n_success when available, otherwise from "
        r"n_failed or finite summary metrics. "
        r"NJ patristic RMSE/Spearman compare pairwise leaf-to-leaf patristic distances "
        r"between the NJ tree reconstructed from the completed matrix and the reference ML tree. "
        r"No fixed-ML-topology fitted-patristic values are used as fallback in these main columns. "
        r"edge BL RMSE is taken from the ML-topology branch-length comparison. "
        r"N/A indicates that the corresponding summary file or metric column was not available.}"
    )
    lines.append(r"\end{sidewaystable}")
    return "\n".join(lines)


def make_latex_audit(df: pd.DataFrame) -> str:
    metrics_for_best = [
        "rmse_miss",
        "mae_miss",
        "pearson_miss",
        "spearman_miss",
        "runtime_seconds",
        "nrf",
        "pat_rmse",
        "edge_bl_rmse",
        "fit_pat_rmse",
        "fit_pat_spearman",
    ]
    best_lookup = {m: best_mask(df, m) for m in metrics_for_best}

    lines: List[str] = []
    lines.append(r"% Requires: \usepackage{booktabs,multirow,rotating}")
    lines.append(r"% Recommended macro: \newcommand{\best}[1]{\textbf{#1}}")
    lines.append(r"\begin{sidewaystable}[t]")
    lines.append(
        r"\caption{Reviewer-audit table: missing-entry reconstruction accuracy, runtime, "
        r"optimization size, convergence diagnostics, success/failure, topology accuracy, "
        r"NJ-vs-ML patristic accuracy, and fixed-ML-topology branch-length diagnostics.}"
    )
    lines.append(r"\label{tab:reviewer_audit_results}")
    lines.append(r"\centering")
    lines.append(r"\tiny")
    lines.append(r"\setlength{\tabcolsep}{1.2pt}")
    lines.append(r"\begin{tabular}{llcccccccccccccc}")
    lines.append(r"\toprule")
    lines.append(
        r"$p$ & Method & RMSE & MAE & Pear. & Spear. & Time(s) & "
        r"$n_{\rm miss}/n_{\rm obs}$ & Vars & Epoch/iter & Succ. & "
        r"RF$_{\rm norm}$ & NJ pat.\ RMSE & edge BL RMSE & fit pat.\ RMSE & fit pat.\ Spear. \\"
    )
    lines.append(r"\midrule")

    for pct in MISSING_LEVELS:
        sub = ordered_subtable(df, pct)
        nrows = len(sub)

        for i, (_, row) in enumerate(sub.iterrows()):
            pcell = rf"\multirow{{{nrows}}}{{*}}{{{pct}\%}}" if i == 0 else ""

            succ = success_cell(row)
            miss_obs = diag_pair_cell(row, "n_missing", "n_observed", decimals=0)
            vars_txt = variables_cell(row)
            epoch_txt = epoch_or_iter_cell(row)

            line = " & ".join([
                pcell,
                row["method_latex"],
                metric_cell(row, "rmse_miss", best_lookup, decimals=2, scale=100.0),
                metric_cell(row, "mae_miss", best_lookup, decimals=2, scale=100.0),
                metric_cell(row, "pearson_miss", best_lookup, decimals=2),
                metric_cell(row, "spearman_miss", best_lookup, decimals=2),
                metric_cell(row, "runtime_seconds", best_lookup, decimals=2),
                miss_obs,
                vars_txt,
                epoch_txt,
                succ,
                metric_cell(row, "nrf", best_lookup, decimals=2),
                metric_cell(row, "pat_rmse", best_lookup, decimals=2, scale=100.0),
                metric_cell(row, "edge_bl_rmse", best_lookup, decimals=2, scale=100.0),
                metric_cell(row, "fit_pat_rmse", best_lookup, decimals=2, scale=100.0),
                metric_cell(row, "fit_pat_spearman", best_lookup, decimals=2),
            ]) + r" \\"
            lines.append(line)

        if pct != MISSING_LEVELS[-1]:
            lines.append(r"\midrule")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\vspace{2pt}")
    lines.append(
        r"\parbox{\textwidth}{\scriptsize "
        r"Errors are computed only on hidden entries. Succ. is displayed as successful/evaluated "
        r"runs in the 30-run protocol. Vars denotes optimized variables for optimization methods and "
        r"imputed entries for direct-imputation baselines. Epoch/iter reports convergence "
        r"epoch, epochs used, iterations, or constructive joins, depending on the method "
        r"summary. NJ pat. RMSE uses only the NJ-vs-reference-ML-tree patristic comparison. "
        r"fit pat. RMSE/Spear. are fixed-ML-topology fitted-patristic diagnostics and are "
        r"not mixed with the main NJ patristic metrics.}"
    )
    lines.append(r"\end{sidewaystable}")
    return "\n".join(lines)


# ============================================================
# Main
# ============================================================

def main() -> None:
    df, inventory = build_combined_table()

    numeric_path = OUT_DIR / "combined_all_metrics_numeric.csv"
    inventory_path = OUT_DIR / "combined_file_inventory.csv"
    main_tex_path = OUT_DIR / "combined_main_results_table.tex"
    audit_tex_path = OUT_DIR / "combined_reviewer_audit_table.tex"

    df.to_csv(numeric_path, index=False)
    inventory.to_csv(inventory_path, index=False)

    main_tex_path.write_text(make_latex_main(df), encoding="utf-8")
    audit_tex_path.write_text(make_latex_audit(df), encoding="utf-8")

    print(f"Saved: {numeric_path}")
    print(f"Saved: {inventory_path}")
    print(f"Saved: {main_tex_path}")
    print(f"Saved: {audit_tex_path}")

    missing_files = inventory[inventory["found"] == False]
    if not missing_files.empty:
        print("\nMissing expected files:")
        for _, r in missing_files.iterrows():
            print(f"  {r['method']:16s} {r['file_type']:8s}: {r['candidate_paths']}")

    # Extra warning: main patristic metrics now require topology NJ-vs-ML columns.
    missing_main_pat = df[
        df["pat_rmse_mean"].isna() | df["pat_spearman_mean"].isna()
    ][["pct_missing", "method_key", "topology_summary_path", "pat_rmse_source_col", "pat_spearman_source_col"]]

    if not missing_main_pat.empty:
        print("\nWarning: some main NJ patristic metrics are N/A.")
        print("This is expected if the topology summary file lacks nj_vs_ML_pat_RMSE or nj_vs_ML_pat_Spearman.")
        for _, r in missing_main_pat.iterrows():
            print(
                f"  p={int(r['pct_missing'])}%  {r['method_key']}: "
                f"topology file = {r['topology_summary_path'] or 'N/A'}"
            )

    print("\nDone.")


if __name__ == "__main__":
    main()


Saved: combined_reviewer_tables/combined_all_metrics_numeric.csv
Saved: combined_reviewer_tables/combined_file_inventory.csv
Saved: combined_reviewer_tables/combined_main_results_table.tex
Saved: combined_reviewer_tables/combined_reviewer_audit_table.tex

This is expected if the topology summary file lacks nj_vs_ML_pat_RMSE or nj_vs_ML_pat_Spearman.
  p=85%  NJstar-STRICT: topology file = NJ-proj-afterrewview/tree_topology_comparison_heterogeneous_primate/tree_topology_summary_meanstd.csv

Done.
